# Phase 4C.1 — Learning Curve Launcher

- **Runtime**: Yêu cầu NVIDIA CUDA GPU tương thích. Tesla T4 là cấu hình tham chiếu. Huấn luyện thực tế hiện mới ghi nhận trên Tesla T4.
- **Đầu vào**: 3 artifacts niêm phong đặt tại `inputs/` trên Google Drive.
- **Thực thi**: Chọn **Run all** để thực hiện toàn bộ 15 runs Stage 1 (frozen).
- **Đầu ra**: Kết quả và log được lưu bền vững vào Google Drive.
- **Lưu ý**: Không chạy đồng thời script operator trong Terminal khi đang chạy notebook.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

EXECUTE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c1")
DRIVE_INPUT_DIR = DRIVE_ROOT / "inputs"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "runs" / "execution_79bb115"

LOCAL_TRANSFER_DIR = Path("/content/forensics-transfer")
LOCAL_OUTPUT_PATH = Path("/content/phase_4c1_outputs")

EXPECTED_ARTIFACTS = {
    "code_archive": {
        "filename": "phase_4c1_code_79bb115.tar.gz",
        "bytes": 758350,
        "sha256": "5e775a7708d1555bf22f56dceb5358e26e07dd224c4b6186f575aff4d2b590d5",
    },
    "reusable_bundle": {
        "filename": "phase_4c1_binary_n250_reusable.tar",
        "bytes": 724633600,
        "sha256": "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27",
    },
    "operator_script": {
        "filename": "phase_4c1_t4_execute_all_stage1.sh",
        "bytes": 49370,
        "sha256": "e16e3e2c7d7aafe3695976062bc86731e5f166a89cc6a350485d13ff9bf500eb",
    },
}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
drive.mount("/content/drive", force_remount=False)

assert sys.platform.startswith("linux"), f"Yêu cầu Linux, nhận được: {sys.platform}"

if EXECUTE:
    assert torch.cuda.is_available(), "CUDA không khả dụng khi EXECUTE=True"
    gpu_model = torch.cuda.get_device_name(0)
    # CUDA Smoke Test
    x = torch.ones((32, 32), device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    assert y.is_cuda and torch.isfinite(y).all(), "CUDA smoke test failed"
    if "T4" not in gpu_model.upper():
        print(f"[!] Cảnh báo: GPU phát hiện là {gpu_model} (tham chiếu: Tesla T4). Tiếp tục với runtime tương thích.")
    else:
        print(f"[+] GPU tham chiếu phát hiện: {gpu_model}")
    
    # Yêu cầu dung lượng đĩa tối thiểu: 2x dataset bundle + 1x code archive + 512 MiB biên an toàn
    SAFETY_MARGIN_BYTES = 512 * 1024 * 1024
    required_bytes = (
        2 * EXPECTED_ARTIFACTS["reusable_bundle"]["bytes"]
        + EXPECTED_ARTIFACTS["code_archive"]["bytes"]
        + SAFETY_MARGIN_BYTES
    )
    content_root = Path("/content") if Path("/content").exists() else Path(".")
    stat_content = shutil.disk_usage(content_root)
    assert stat_content.free >= required_bytes, (
        f"Dung lượng đĩa {content_root} không đủ: "
        f"{stat_content.free / (1024 ** 3):.2f} GiB khả dụng, "
        f"cần tối thiểu {required_bytes / (1024 ** 3):.2f} GiB "
        f"(tính theo 2x bundle + code archive + 512 MiB biên an toàn)."
    )

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục dự án đã chuẩn bị: {DRIVE_ROOT}")
if not DRIVE_INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục inputs đã chuẩn bị: {DRIVE_INPUT_DIR}")

missing_artifacts = []
for key, spec in EXPECTED_ARTIFACTS.items():
    fp = DRIVE_INPUT_DIR / spec["filename"]
    if not fp.exists():
        missing_artifacts.append(spec["filename"])
        continue
    assert fp.stat().st_size == spec["bytes"], f"Sai số byte: {spec['filename']}"
    assert sha256_file(fp) == spec["sha256"], f"Sai SHA-256: {spec['filename']}"

if missing_artifacts:
    actual_entries = sorted(p.name for p in DRIVE_INPUT_DIR.iterdir())
    raise FileNotFoundError(
        f"Thiếu artifact trong {DRIVE_INPUT_DIR}: {missing_artifacts}. "
        f"Nội dung hiện có: {actual_entries}"
    )

LOCAL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)
for key, spec in EXPECTED_ARTIFACTS.items():
    src = DRIVE_INPUT_DIR / spec["filename"]
    dst = LOCAL_TRANSFER_DIR / spec["filename"]
    if dst.exists() and dst.stat().st_size == spec["bytes"] and sha256_file(dst) == spec["sha256"]:
        continue
    part_dst = LOCAL_TRANSFER_DIR / f"{spec['filename']}.part"
    if part_dst.exists():
        part_dst.unlink()
    shutil.copy2(src, part_dst)
    assert part_dst.stat().st_size == spec["bytes"], f"Lỗi copy {part_dst.name}"
    assert sha256_file(part_dst) == spec["sha256"], f"Lỗi SHA-256 {part_dst.name}"
    os.replace(part_dst, dst)

(LOCAL_TRANSFER_DIR / "phase_4c1_t4_execute_all_stage1.sh").chmod(0o755)

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_OUTPUT_PATH.exists() or LOCAL_OUTPUT_PATH.is_symlink():
    if LOCAL_OUTPUT_PATH.is_symlink():
        target_resolved = os.path.realpath(LOCAL_OUTPUT_PATH)
        expected_resolved = os.path.realpath(DRIVE_OUTPUT_DIR)
        assert target_resolved == expected_resolved, f"Symlink trỏ sai: {target_resolved}"
    else:
        raise RuntimeError(f"{LOCAL_OUTPUT_PATH} là thư mục vật lý, không phải symlink")
else:
    LOCAL_OUTPUT_PATH.symlink_to(DRIVE_OUTPUT_DIR, target_is_directory=True)

print("[READY] Preflight và staging hoàn tất")

In [ ]:
if not EXECUTE:
    print("[SKIP] EXECUTE=False: Bỏ qua thực thi huấn luyện")
else:
    print("[RUN] Bắt đầu 15 Stage 1 runs")
    op_script = LOCAL_TRANSFER_DIR / "phase_4c1_t4_execute_all_stage1.sh"
    try:
        subprocess.run(
            ["bash", str(op_script)],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"\n[-] FATAL: Operator thất bại với mã lỗi {e.returncode}")
        failure_json = LOCAL_OUTPUT_PATH / "OPERATOR_FAILURE.json"
        console_log = LOCAL_OUTPUT_PATH / "logs" / "operator_console.log"
        print(f"[*] Persistent console log: {console_log}")
        if failure_json.is_file():
            print("=" * 50)
            print("[*] NỘI DUNG OPERATOR_FAILURE.json:")
            print("=" * 50)
            print(failure_json.read_text(encoding="utf-8"))
        if console_log.is_file():
            log_lines = console_log.read_text(encoding="utf-8", errors="replace").splitlines()
            tail_lines = log_lines[-100:]
            print("=" * 50)
            print(f"[*] 100 DÒNG CUỐI CONSOLE LOG ({len(tail_lines)}/{len(log_lines)}):")
            print("=" * 50)
            for line in tail_lines:
                print(line)
        raise RuntimeError(f"Operator execution failed with exit code {e.returncode}") from e

In [ ]:
REQUIRED_ARCHIVES = [
    "n50_results.tar.gz",
    "n100_results.tar.gz",
    "n250_results.tar.gz",
    "phase_4c1_t4_execution_logs.tar.gz",
    "phase_4c1_all_15_runs_results.tar.gz",
]

if not EXECUTE:
    print("[SKIP] EXECUTE=False: Bỏ qua kiểm tra kết quả")
else:
    drive_download_dir = DRIVE_OUTPUT_DIR / "download"
    assert drive_download_dir.exists(), f"Thiếu download dir: {drive_download_dir}"

    for name in REQUIRED_ARCHIVES:
        archive_path = drive_download_dir / name
        sidecar_path = drive_download_dir / f"{name}.sha256"
        assert archive_path.exists(), f"Thiếu archive: {archive_path}"
        assert sidecar_path.exists(), f"Thiếu sidecar: {sidecar_path}"
        assert archive_path.stat().st_size > 0, f"Archive rỗng: {archive_path}"
        expected_hash = sidecar_path.read_text(encoding="utf-8").strip().split()[0].lower()
        assert sha256_file(archive_path) == expected_hash, f"Sai SHA-256 archive: {name}"

    sample_sizes = (50, 100, 250)
    seeds = (42, 1337, 2025, 3407, 9001)
    completed_runs = 0

    for size in sample_sizes:
        for seed in seeds:
            run_dir = DRIVE_OUTPUT_DIR / f"n{size}_seed_{seed}"
            receipt_path = run_dir / "run_receipt.json"
            assert receipt_path.exists(), f"Thiếu receipt: {run_dir.name}"
            with open(receipt_path, "r", encoding="utf-8") as f:
                receipt = json.load(f)
            assert receipt.get("status") == "completed", f"Trạng thái chưa hoàn thành: {run_dir.name}"
            assert receipt.get("sample_size") == size, f"Sai sample_size: {run_dir.name}"
            assert receipt.get("seed") == seed, f"Sai seed: {run_dir.name}"
            assert receipt.get("stage") == "frozen", f"Sai stage: {run_dir.name}"
            assert receipt.get("locked_test_access", -1) == 0, f"Locked-test access vi phạm: {run_dir.name}"
            assert receipt.get("stage2_invocations", -1) == 0, f"Stage 2 vi phạm: {run_dir.name}"
            assert receipt.get("validation_source_count") == 91, f"Sai số validation sources: {run_dir.name}"
            completed_runs += 1

    assert completed_runs == 15, f"Kỳ vọng 15 runs, nhận được: {completed_runs}"
    print("[PASS] Hoàn tất và xác thực 15/15 runs")